# DICOM HTJ2K Transcoder

GPU-accelerated DICOM pipeline using the `htj2k_transcoder` library.

**What it does:**
1. **Transcodes** pixel data to **HTJ2K** (lossless) via NVIDIA nvImageCodec
2. **Converts** legacy single-frame series to **Enhanced Multi-Frame** DICOM via highdicom
3. **Distributes** work across multiple GPUs with **Ray** actors
4. **Tracks progress** incrementally via **Delta** tables

**Two-phase architecture** (Spark Connect compatible):
- **Phase 1** (`foreachBatch`): Discover files, scan DICOM headers, MERGE as `pending` into results table
- **Phase 2** (notebook kernel): Read `pending` rows, dispatch to Ray GPU actors, persist results

## 1. Install Dependencies

In [ ]:
%pip install pydicom>=3.0.0 pylibjpeg>=2.0.0 pylibjpeg-openjpeg>=2.0.0 pyjpegls>=1.2
%pip install ray[default] torch
%pip install nvidia-nvimgcodec-cu12[all]
%pip install highdicom

## 2. Stage nvImageCodec DICOM tools

Copy the nvimgcodec DICOM tools into the installed package so they can be imported.

In [ ]:
import shutil
import nvidia.nvimgcodec
import os

dst_dir = os.path.dirname(nvidia.nvimgcodec.__file__)
os.makedirs(dst_dir + "/tools/", exist_ok=True)
shutil.copytree("nvimgcodec/tools/", dst_dir + "/tools/", dirs_exist_ok=True)

from nvidia.nvimgcodec.tools.dicom import convert_htj2k
print('✓ nvImageCodec DICOM tools installed')

In [ ]:
dbutils.library.restartPython()

## 3. Verify Environment

In [ ]:
import subprocess, sys
import pydicom, ray, torch, numpy as np
from packaging.version import parse

print(f"pydicom:  {pydicom.__version__}  {'✓' if parse(pydicom.__version__) >= parse('3.0.0') else '⚠️ need >=3.0.0'}")
print(f"ray:      {ray.__version__}")
print(f"torch:    {torch.__version__}")
print(f"numpy:    {np.__version__}")

try:
    from nvidia import nvimgcodec
    print(f"nvimgcodec: {nvimgcodec.__version__}  ✓")
except ImportError:
    print("nvimgcodec: NOT FOUND  ⚠️")

try:
    import highdicom
    print(f"highdicom: {highdicom.__version__}  ✓")
except ImportError:
    print("highdicom: NOT FOUND  ⚠️")

try:
    from nvidia.nvimgcodec.tools.dicom.convert_htj2k import transcode_datasets_to_htj2k
    from nvidia.nvimgcodec.tools.dicom.convert_multiframe import convert_to_enhanced_dicom
    from nvidia.nvimgcodec.tools.dicom.dicom_utils import DicomSeriesScanner, DicomFileLoader
    print("nvimgcodec DICOM tools:  ✓")
except ImportError as e:
    print(f"nvimgcodec DICOM tools: FAILED  ⚠️  {e}")

print(f"\nGPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=False)

## 4. Configuration

Import the library and set up pipeline configuration. Edit the values below for your environment.

In [ ]:
from htj2k_transcoder import (
    TranscodeConfig, MergeConfig, InputConfig, StreamingConfig,
    create_results_table,
    start_autoloader_stream, start_delta_stream,
    transcode_pending_series,
    discover_and_group, run_batch,
    print_summary,
)

# ============================================================
# Paths — edit these for your environment
# ============================================================
DICOM_ROOT_DIR = "/Volumes/ema_rina/pixels_solacc_tcia/pixels_volume/unzipped/"
OUTPUT_DIR     = "/Volumes/ema_rina/pixels_solacc_tcia/pixels_volume/htj2k_multiframe_nvc/"

# ============================================================
# GPU / Ray tuning
# ============================================================
NUM_GPUS           = 8
ACTORS_PER_GPU     = 4
BATCH_SIZE         = NUM_GPUS * 2      # series per actor __call__
NUM_CPUS_PER_ACTOR = 3
IO_POOL_SIZE       = BATCH_SIZE * 4

# ============================================================
# Pipeline configuration
# ============================================================
TRANSCODE = TranscodeConfig(
    enabled=True,
    num_resolutions=6,
    code_block_size=(64, 64),
    progression_order="RPCL",   # -> Transfer Syntax 1.2.840.10008.1.2.4.202
    max_batch_size=256,
)

MERGE = MergeConfig(
    enabled=True,               # Enhanced Multi-Frame (CT/MR/PT via highdicom)
)

INPUT = InputConfig(
    source="DELTA",             # "DIRECTORY" for Auto Loader, "DELTA" for Delta streaming
    delta_table="ema_rina.pixels_solacc_tcia.object_catalog",
    filter_encoded=True,        # skip files already in HTJ2K
)

STREAMING = StreamingConfig(
    enabled=True,
    checkpoint_path="/Volumes/ema_rina/pixels_solacc_tcia/pixels_volume/_checkpoints/nvc_htj2k/",
    results_table="ema_rina.pixels_solacc_tcia.htj2k_nvc_results",
    trigger="availableNow",
    max_files_per_trigger=10000,
)

# ============================================================
# Print active configuration
# ============================================================
print(f"Root dir:   {DICOM_ROOT_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"GPUs:       {NUM_GPUS}")
print(f"Transcode:  enabled={TRANSCODE.enabled}, progression={TRANSCODE.progression_order}")
print(f"Merge:      enabled={MERGE.enabled} (highdicom Enhanced CT/MR/PT)")
print(f"Input:      source={INPUT.source}")
print(f"Streaming:  enabled={STREAMING.enabled}")
print(f"Pipeline:   batch={BATCH_SIZE}, cpus/actor={NUM_CPUS_PER_ACTOR}, pool={IO_POOL_SIZE}")

## 5. Phase 1 — Ingest & Stage Pending Series

Phase 1 scans DICOM headers, groups by series, and MERGEs them as `pending` into the results table.

**Choose one** of the two cells below depending on your `INPUT.source`:
- **Cell A**: Auto Loader (for `DIRECTORY` source — cloud file discovery)
- **Cell B**: Delta Streaming (for `DELTA` source — reads from an existing object catalog table)

In [ ]:
# --- Cell A: Auto Loader (DIRECTORY source) ---
# Uncomment and run this cell if INPUT.source == "DIRECTORY"

# assert INPUT.source == "DIRECTORY", "This cell is for DIRECTORY source"
# start_autoloader_stream(spark, DICOM_ROOT_DIR, STREAMING)

In [ ]:
# --- Cell B: Delta Streaming (DELTA source) ---
assert INPUT.source == "DELTA", "This cell is for DELTA source. Use Auto Loader cell for DIRECTORY."

start_delta_stream(spark, INPUT, STREAMING)

In [ ]:
spark.table(STREAMING.results_table).groupBy("status").count().orderBy("status").display()

## 6. Phase 2 — GPU Processing

Reads `pending` rows from the results table, dispatches to Ray GPU actors with prefetch I/O overlap, writes results back via Delta MERGE.

In [ ]:
transcode_pending_series(
    spark,
    results_table=STREAMING.results_table,
    output_dir=OUTPUT_DIR,
    transcode_cfg=TRANSCODE,
    merge_cfg=MERGE,
    num_gpus=NUM_GPUS,
    batch_size=BATCH_SIZE,
    actors_per_gpu=ACTORS_PER_GPU,
    num_cpus_per_actor=NUM_CPUS_PER_ACTOR,
    io_pool_size=IO_POOL_SIZE,
)

## 7. Results

In [ ]:
display(spark.table(STREAMING.results_table))

## 8. Summary

In [ ]:
summary_df = print_summary(
    spark,
    results_table=STREAMING.results_table,
    transcode_cfg=TRANSCODE,
    merge_cfg=MERGE,
    num_gpus=NUM_GPUS,
)
display(summary_df)

---

## Alternative: Batch Mode (Non-Streaming)

For one-shot processing without Structured Streaming. Uses `DicomSeriesScanner` from nvImageCodec to walk the directory and group by series, then dispatches to Ray.

In [ ]:
# --- Batch mode (alternative to streaming) ---
# Uncomment to use:

# results = run_batch(
#     root_dir=DICOM_ROOT_DIR,
#     output_dir=OUTPUT_DIR,
#     transcode_cfg=TRANSCODE,
#     merge_cfg=MERGE,
#     num_gpus=NUM_GPUS,
# )
#
# # Convert results to DataFrame for inspection
# results_df = spark.createDataFrame(results)
# display(results_df)